# 🤖 하나봇 파인튜닝 (Google Colab)

Unsloth + Qwen2.5-3B-Instruct (4-bit)로 하나봇 모델을 파인튜닝합니다.

## 1️⃣ GPU 확인

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')

## 2️⃣ Unsloth 설치 (3-5분 소요)

In [ ]:
!pip install unsloth[colab-new] -q
!pip install datasets transformers bitsandbytes xformers -q
print('✓ 설치 완료')

## 3️⃣ 모델 로드 (Qwen2.5-3B-Instruct-bnb-4bit)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct-bnb-4bit',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
print('✓ 모델 로드 완료')

## 4️⃣ LoRA 설정 (r=16, lora_alpha=16)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing=True,
    use_rslora=True,
)
print('✓ LoRA 적용 완료')

## 5️⃣ 학습 데이터 업로드

In [ ]:
from google.colab import files

print('hanabot_train.jsonl 파일을 선택해주세요')
uploaded = files.upload()
train_file = list(uploaded.keys())[0]
print(f'✓ 업로드됨: {train_file}')

## 6️⃣ 데이터 로드 및 형식 확인

In [ ]:
import json
from datasets import Dataset

# 데이터 로드
data = []
with open(train_file, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

# 형식 확인
print(f'총 {len(data)}개 데이터')
print(f'예시: {data[0]}')

# Dataset 변환
dataset = Dataset.from_dict({
    'text': [str(item['conversations']) for item in data]
})

## 7️⃣ 학습 설정

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=50,
    output_dir='outputs',
    optim='paged_adamw_8bit',
    seed=42,
)
print('✓ 학습 파라미터 설정 완료')

## 8️⃣ 학습 실행 (20-30분 소요)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=2048,
)

trainer.train()
print('✓ 학습 완료')

## 9️⃣ Loss 그래프 (학습 과정 시각화)

In [ ]:
import matplotlib.pyplot as plt
import json

# 학습 로그에서 loss 추출
log_file = 'outputs/trainer_state.json'
with open(log_file) as f:
    logs = json.load(f)

steps = [l['step'] for l in logs['log_history'] if 'loss' in l]
losses = [l['loss'] for l in logs['log_history'] if 'loss' in l]

plt.figure(figsize=(10, 5))
plt.plot(steps, losses, marker='o')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Learning Curve')
plt.grid()
plt.show()

## 🔟 평가 (학습 전후 비교)

In [ ]:
# 평가 데이터 업로드
print('hanabot_eval.jsonl 파일을 선택해주세요')
uploaded = files.upload()
eval_file = list(uploaded.keys())[0]

# 평가 샘플 추출
eval_data = []
with open(eval_file, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i < 3:  # 처음 3개만
            eval_data.append(json.loads(line))

print(f'평가 샘플: {len(eval_data)}개')
for i, item in enumerate(eval_data, 1):
    q = item['conversations'][0]['content']
    a = item['conversations'][1]['content']
    print(f'\n[샘플 {i}] Q: {q}')
    print(f'정답: {a[:100]}...')

## 1️⃣1️⃣ 모델 내보내기 (GGUF q4_k_m)

In [ ]:
from unsloth import unsloth_to_gguf

unsloth_to_gguf(
    model,
    tokenizer,
    quantization_method='q4_k_m',
    output_filename='hanabot_finetuned.gguf',
)
print('✓ GGUF 변환 완료')

## 1️⃣2️⃣ Modelfile 생성 (Ollama 호환)

In [ ]:
modelfile = '''FROM ./hanabot_finetuned.gguf
PARAMETER temperature 0.7
PARAMETER top_p 0.9
SYSTEM \"당신은 하나 AI 학원의 도우미 하나봇입니다. 학원 규정에 대한 질문에 답합니다.\"'''

with open('Modelfile', 'w') as f:
    f.write(modelfile)

print('✓ Modelfile 생성 완료')

## 1️⃣3️⃣ 파일 압축 & 다운로드

In [ ]:
import zipfile
import os

# zip 파일 생성
with zipfile.ZipFile('hanabot_finetuned.zip', 'w') as z:
    z.write('hanabot_finetuned.gguf')
    z.write('Modelfile')

print(f'압축 크기: {os.path.getsize("hanabot_finetuned.zip") / 1e9:.1f}GB')
print('✓ hanabot_finetuned.zip 준비 완료')
files.download('hanabot_finetuned.zip')